Fonte dos dados - https://dados.gov.br/dados/conjuntos-dados/atendimentos-realizados-na-central-de-atendimento-da-conta-govbr

##### Objetivos

1. Padronizar os nomes das colunas
2. Ler os arquivos CSV
3. Converter CSV para Parquet, reduzindo o tamanho do dataset
4. Configurar autenticação no Google Cloud
5. Definir um Schema para receber os dados no BigQuery
6. Carregar os dados no BigQuery

##### Imports

In [ ]:
import os
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import tqdm

##### Padronizar os nomes das colunas

In [ ]:
# Nomes das colunas
colunas = ['solicitacao',
           'origem',
           'categoria_solucao','datainicio',
           'atendimentoinicio',	
           'encerramento',
           'grupo',	
           'situacao'
           ]

##### Função que lê cada CSV na pasta, ajusta os tipos das colunas e converte em novo arquivo parquet

In [ ]:
# Converter os arquivos da pasta para o formato parquet
def convert_parquet(caminho_pasta, separador):
    for file in tqdm.tqdm(os.listdir(path=caminho_pasta)):
        if file.endswith('.csv'):
            df = pd.read_csv(filepath_or_buffer= f'{caminho_pasta}/{file}',
                encoding='utf-8',
                sep=separador,
                skiprows=1,
                names=colunas         
                )
            # Converte colunas de data para datetime
            df['datainicio']        = pd.to_datetime(df['datainicio'], errors='coerce')
            df['atendimentoinicio'] = pd.to_datetime(df['atendimentoinicio'], errors='coerce')
            df['encerramento']      = pd.to_datetime(df['encerramento'], errors='coerce')

            # Converte colunas de texto para string
            df['solicitacao']       = df['solicitacao'].astype(str)
            df['origem']            = df['origem'].astype(str)
            df['categoria_solucao'] = df['categoria_solucao'].astype(str)
            df['grupo']             = df['grupo'].astype(str)
            df['situacao']          = df['situacao'].astype(str)
            
            table = pa.Table.from_pandas(df)                                    # Salva o df em uma table temporária
            parquet_file = f'{caminho_pasta}/{file.replace('.csv','.parquet')}' # Cria um parquet_file para receber os dados
            pq.write_table(table, parquet_file)                                 # Escreve os dados da table temporária no parquet_file
            print(f'Arquivo {file} convertido') 

In [ ]:
# Pasta com os arquivos CSV dos meses 01 a 04 (pasta1) - CSV sep=','
pasta1 = r"C:\Users\Jorge Gabriel\OneDrive\Portfólio Analista de Dados\Central GOV BR - Indicadores de Atendimento\data\raw\pasta 1"

# Pasta com os arquivos CSV dos meses 05 a 12 (pasta2) - CSV sep=';'
pasta2 = r"C:\Users\Jorge Gabriel\OneDrive\Portfólio Analista de Dados\Central GOV BR - Indicadores de Atendimento\data\raw\pasta 2"

In [ ]:
convert_parquet(pasta1, ',')
convert_parquet(pasta2, ';')

##### Carga dos arquivos Parquet para o BigQuery

In [ ]:
# Imports
import os
from google.cloud import bigquery
from google.oauth2 import service_account

# Configurações de Autenticação
pasta_parquet = r'C:\Users\Jorge Gabriel\OneDrive\Portfólio Analista de Dados\Central GOV BR - Indicadores de Atendimento\Raw_Data\Parquet'

credencial_gcp = r'C:\Users\Jorge Gabriel\OneDrive\Portfólio Analista de Dados\Central GOV BR - Indicadores de Atendimento\API BigQuery Key\govbr-atendimentos-d2d3843292dc.json'

project_id = 'govbr-atendimentos'
dataset_id = 'bronze_dataset'
table_id = 'bronze_tabela'

table_full_id = f"{project_id}.{dataset_id}.{table_id}"


# Autenticação
credentials = service_account.Credentials.from_service_account_file(credencial_gcp)
client = bigquery.Client(credentials=credentials, project=project_id)


# Fixar Schema
schema = [
    bigquery.SchemaField("solicitacao", "STRING"),
    bigquery.SchemaField("origem", "STRING"),
    bigquery.SchemaField("categoria_solucao", "STRING"),
    bigquery.SchemaField("datainicio", "DATETIME"),  # campo de partição
    bigquery.SchemaField("atendimentoinicio", "DATETIME"),
    bigquery.SchemaField("encerramento", "DATETIME"),
    bigquery.SchemaField("grupo", "STRING"),
    bigquery.SchemaField("situacao", "STRING")
]


# Configuração do Job
job_config = bigquery.LoadJobConfig(
    source_format=bigquery.SourceFormat.PARQUET,
    write_disposition="WRITE_APPEND",
    schema=schema  # schema fixo (sem autodetect)
)


# Loop de Carga
for file in os.listdir(pasta_parquet):
    if file.endswith(".parquet"):

        file_path = os.path.join(pasta_parquet, file)
        print(f"Enviando: {file}")

        with open(file_path, "rb") as f:
            load_job = client.load_table_from_file(
                f,
                table_full_id,
                job_config=job_config
            )

        load_job.result()
        print(f"{file} carregado")

print("Carga finalizada com sucesso")
